# RDSS Temporal Common-State SHADOW Gate v0.1

TMR is the fixed default. This notebook first verifies exact token-level non-interference while collecting common-state SHADOW observations. Counterfactual continuations are development-only.


In [ ]:
# User-editable campaign variables
SEED = 14
LIMIT = 10
CAMPAIGN_PROMPT_IDS = ''  # empty = full campaign; use 'p02' for targeted development replication
RUN_DEVELOPMENT_COUNTERFACTUALS = False
DEVELOPMENT_PROMPT_IDS = 'p01,p02,p03,p04,p05,p06'


In [ ]:
# Reproducible Qwen3.5 runtime bootstrap (official Qwen requirement: current Transformers main)
import subprocess, sys
TRANSFORMERS_COMMIT = '96331a9f93b72697f160a958d2883d4b49a56739'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
    f'git+https://github.com/huggingface/transformers.git@{TRANSFORMERS_COMMIT}',
    'bitsandbytes>=0.46.1',
], check=True)
print('Pinned Transformers commit:', TRANSFORMERS_COMMIT)


In [ ]:
import base64, json
from pathlib import Path

EMBEDDED = json.loads("{\"PROTOCOL.md\":\"IyBSRFNTIFRlbXBvcmFsIENvbW1vbi1TdGF0ZSBTSEFET1cgR2F0ZSDigJQgcHJvdG9jb2wgdjAuMQoKIyMgUXVlc3Rpb24KCkNhbiBSRFNTIHJlY292ZXIgdGhlIHNtYWxsIHJlc2lkdWFsIGhlYWRyb29tIGxlZnQgYnkgZml4ZWQgVEFMT04gbWFzcy1yaWdodAooVE1SKSB3aXRob3V0IGRlZ3JhZGluZyBUTVIncyBkb21pbmFudCBwZXJmb3JtYW5jZSBhZnRlciBjb21wdXRlIGFuZCBoYXJtIGFyZQpjaGFyZ2VkPwoKVGhpcyBpcyBub3QgYSBjb250ZXN0IGJldHdlZW4gVEFMTSBhbmQgVEFMT04uIFRNUiBpcyB0aGUgZGVmYXVsdCBhdXRob3JpdHkuClRoZSBnYXRlIHN0dWRpZXMgb25seSB3aGVuIGV2aWRlbmNlIGlzIHN0cm9uZyBlbm91Z2ggdG8gbGVhdmUgaXQuCgojIyBDYXVzYWwgY29udHJhY3QKCjEuIEdlbmVyYXRlIHRoZSBwcmVmaXggdW5kZXIgVE1SLgoyLiBBdCBmcm96ZW4gY2hlY2twb2ludHMsIGNsb25lIHRoZSBzYW1lIHJhdyBtb2RlbCBsb2dpdHMgYW5kIHByZWZpeC4KMy4gQXBwbHkgZWFjaCByZXNjdWUgb3BlcmF0b3IgaW4gU0hBRE9XLiBBIHByb2JlIHNhbXBsZXMgbm8gdG9rZW4sIGNvbnN1bWVzIG5vCiAgIGdlbmVyYXRpb24gUk5HLCBhbmQgbXV0YXRlcyBuZWl0aGVyIHRoZSBUTVIgcHJvY2Vzc29yIG5vciB0aGUgcHJlZml4Lgo0LiBJbiBzdGFnZSBHMWEgKGBTSEFET1dfT05MWWApLCBnZW5lcmF0aW9uIGFsd2F5cyByZW1haW5zIHVuZGVyIFRNUi4gRXhhY3QKICAgdG9rZW4gaWRlbnRpdHkgd2l0aCB0aGUgZml4ZWQtVE1SIGFybSBpcyBtYW5kYXRvcnkuCjUuIE9uIGRldmVsb3BtZW50IHByb21wdHMgb25seSwgbGF1bmNoIGNvdW50ZXJmYWN0dWFsIGNvbnRpbnVhdGlvbnMgZnJvbSB0aGUKICAgY2FwdHVyZWQgY29tbW9uIHByZWZpeCB0byBsYWJlbCByZXNpZHVhbCBoZWFkcm9vbSBhbmQgZml0IHRoZSBjaGVhcCBzY3JlZW4KICAgcGx1cyB2YWx1ZSBtb2RlbC4KNi4gRnJlZXplIGhhc2hlcywgY29lZmZpY2llbnRzLCB0aHJlc2hvbGRzLCBzcGxpdCwgYW5kIHJlc2N1ZSBsYWRkZXIuCjcuIEV2YWx1YXRlIG9uY2Ugb24gdGhlIHByb3RlY3RlZCByZXNlcnZlLiBQb3N0LWhvYyBvcmFjbGUgcmVzdWx0cyBhcmUKICAgZGVzY3JpcHRpdmUgYW5kIGNhbm5vdCBhdXRob3JpemUgYSBzd2l0Y2guCgojIyBBdXRob3JpdHkgbGFkZGVyCgpgdGFsb25fbWFzc19yaWdodGAgaXMgdGhlIGZpeGVkIGRlZmF1bHQuIFRoZSBvcmRlcmVkIHJlc2N1ZSBzZXQgaXM6CgoxLiBgdGFsb25fbWFzc19sZWZ0YAoyLiBgdGFsbV9tYXNzX3JpZ2h0X3NvZnRgCjMuIGBiYXNlbGluZWAKCmBBQlNUQUlOYCBhbmQgYEFTS2AgYXJlIGZvcm1hbCBvdXRwdXRzLiBNaXNzaW5nLCBpbW1hdHVyZSwgb3V0LW9mLWRpc3RyaWJ1dGlvbiwKb3IgdW5zYWZlIGV2aWRlbmNlIGZhaWxzIGNsb3NlZCB0byBUTVIuCgojIyBBcm1zCgp8IEFybSB8IFB1cnBvc2UgfCBNYXkgYWZmZWN0IHRva2Vucz8gfAp8LS0tfC0tLXwtLS06fAp8IEZpeGVkIFRNUiB8IERvbWluYW50IGJhc2VsaW5lIHwgWWVzLCBUTVIgb25seSB8CnwgVE1SICsgU0hBRE9XIHwgTm9uLWludGVyZmVyZW5jZSBhbmQgZmVhdHVyZSBjYXB0dXJlIHwgTm8gcmVsYXRpdmUgdG8gZml4ZWQgVE1SIHwKfCBDaGVhcCBzY3JlZW4gb25seSB8IFNjcmVlbiB2YWx1ZSBhbmQgY2FsaWJyYXRpb24gfCBPbmx5IGFmdGVyIGZyZWV6ZSB8CnwgU2NyZWVuICsgU0hBRE9XIHwgUHJpbWFyeSBSRFNTIHBvbGljeSB8IE9ubHkgYWZ0ZXIgZnJlZXplIHwKfCBQcm9iZS1hbGwgU0hBRE9XIHwgQ29tcHV0ZS1jb3N0IHVwcGVyIGNvbnRyb2wgfCBObyBpbiBHMWEgfAp8IENvdmVyYWdlLW1hdGNoZWQgcmFuZG9tIHwgVHJpZ2dlciBjb250cm9sIHwgQWZ0ZXIgZnJlZXplIHwKfCBQb3N0LWhvYyBvcmFjbGUgfCBSZXNpZHVhbC1oZWFkcm9vbSBjZWlsaW5nIHwgTmV2ZXIgfAoKIyMgUHJpbWFyeSBlc3RpbWFuZAoKRm9yIGEgZnJvemVuIHJlc2N1ZSBjYW5kaWRhdGUgXChhXCkgYXQgaGlzdG9yeSBcKEhfdFwpOgoKXFsKXG9wZXJhdG9ybmFtZXtWb0N9X3tMQ0J9KGEsIEhfdCkgPQpMQ0Jfezk1fVtVKGEpLVUoVE1SKVxtaWQgSF90XSAtIENfe3Byb2JlfQotIFxsYW1iZGFfUlwsVUNCX3s5NX1bUChcbWF0aHJte2hhcm19KV0uClxdCgpTd2l0Y2ggb25seSB3aGVuIHRoZSBjaGVhcCBzY3JlZW4gaXMgcG9zaXRpdmUsIHRoZSBwb2xpY3kgaXMgZnJvemVuLApgVm9DX0xDQiA+IDBgLCBhbmQgdGhlIGhhcm0gVUNCIGlzIG5vIGdyZWF0ZXIgdGhhbiB0aGUgcHJlZGVjbGFyZWQgZGVsdGEuCgojIyBQcm9tb3Rpb24gZ2F0ZQoKVGhlIGdhdGUgcGFzc2VzIG9ubHkgaWYgYWxsIGNvbmRpdGlvbnMgaG9sZCBvbiB0aGUgcHJvdGVjdGVkIHJlc2VydmU6CgotIGZpeGVkIFRNUiBhbmQgU0hBRE9XLW9ubHkgb3V0cHV0cyBhcmUgdG9rZW4taWRlbnRpY2FsOwotIHBhaXJlZCB1dGlsaXR5IG9mIHNjcmVlbitTSEFET1cgZXhjZWVkcyBmaXhlZCBUTVIgYWZ0ZXIgY2hhcmdlZCBjb3N0OwotIHRoZSBsb3dlciBjb25maWRlbmNlIGJvdW5kIG9mIHRoZSBpbXByb3ZlbWVudCBpcyBwb3NpdGl2ZTsKLSBoYXJtIFVDQjk1IGlzIHdpdGhpbiB0aGUgY29uZmlndXJlZCBkZWx0YTsKLSBpbXByb3ZlbWVudCBpcyBub3QgZXhwbGFpbmVkIGJ5IGEgc2luZ2xlIHByb21wdCwgc2VlZCwgb3IgYmVuY2htYXJrIGZhbWlseTsKLSBhbGwgZGVjaXNpb25zIGFyZSByZWNvbnN0cnVjdGlibGUgZnJvbSBmcm96ZW4gbG9ncyBhbmQgaGFzaGVzLgoKVGhlIDEwLXByb21wdCDDlyAzLXNlZWQgaGlzdG9yaWNhbCBhcmNoaXZlIGlzIGEgZmVhc2liaWxpdHkgc291cmNlIG9ubHkuIEl0CmNhbm5vdCBwcm9tb3RlIHRoZSBnYXRlIGJlY2F1c2UgaXRzIGlubGluZSB0cmFjZXMgd2VyZSBwcm9kdWNlZCBieSBhIGRpZmZlcmVudCwKZmxhdCByb3V0ZXIgYW5kIGFyZSBub3QgY29tbW9uLXN0YXRlIFRNUiBvYnNlcnZhdGlvbnMuCgojIyBMaW5lYWdlIHJ1bGUKClRoZSBleGVjdXRhYmxlIGFkYXB0ZXIgaW4gdGhpcyByZWxlYXNlIHJlcHJvZHVjZXMgdGhlIGBoaXN0b3JpY2FsX3YwNGAgb3BlcmF0b3IKbGluZWFnZTogb3BlcmF0b3IgdGVtcGVyYXR1cmUgMS4wLCBpbnRlcm5hbCBgcHJlc2VydmVfdG9wMT1mYWxzZWAsIGZvbGxvd2VkIGJ5Cm9uZSBjb21tb24gZXh0ZXJuYWwgdG9wLTEgcHJvamVjdGlvbi4gVGhlIHNlcGFyYXRlIGNhbm9uaWNhbCBUTVIgY29uZmlndXJhdGlvbgp3aXRoIGhhc2ggYDY3ODc1ZjdlLi4uMTJhYCBpcyByZWNvcmRlZCBmb3IgcmVmZXJlbmNlIGFuZCBtdXN0IGJlIGV2YWx1YXRlZCBhcyBhCnNlcGFyYXRlIGFybS4gUmVzdWx0cyBmcm9tIHRoZSB0d28gbGluZWFnZXMgbXVzdCBuZXZlciBiZSBwb29sZWQgc2lsZW50bHkuCg==\",\"config/gate_v0_1.json\":\"ewogICJnYXRlX25hbWUiOiAiUkRTUyBUZW1wb3JhbCBDb21tb24tU3RhdGUgU0hBRE9XIEdhdGUiLAogICJ2ZXJzaW9uIjogIjAuMS4wIiwKICAic2NpZW50aWZpY19zdGF0ZSI6ICJQUk9TUEVDVElWRV9HUFVfUEVORElORyIsCiAgIm1vZGUiOiAic2hhZG93X29ubHkiLAogICJhY3RpdmVfb3BlcmF0b3JfbGluZWFnZSI6IHsKICAgICJuYW1lIjogImhpc3RvcmljYWxfdjA0X2NvbW1vbl9wcm9qZWN0aW9uIiwKICAgICJzb3VyY2UiOiAiY2ZsZHJfcGx1c192MF80X3JlcGFpcmVkLnB5IiwKICAgICJ0YWxvbl9tYXNzX3JpZ2h0IjogewogICAgICAic3RyZW5ndGgiOiAwLjcsCiAgICAgICJ0ZW1wZXJhdHVyZSI6IDEuMCwKICAgICAgIm1hc3NfdGhyZXNob2xkIjogMC45MiwKICAgICAgInNlbGVjdGlvbl9tb2RlIjogIm1hc3MiLAogICAgICAic2lkZV9wb2xpY3kiOiAicmlnaHQiLAogICAgICAicHJlc2VydmVfdG9wMV9pbnNpZGVfb3BlcmF0b3IiOiBmYWxzZSwKICAgICAgImNvbW1vbl9leHRlcm5hbF90b3AxX3Byb2plY3Rpb24iOiB0cnVlCiAgICB9CiAgfSwKICAiY2Fub25pY2FsX3Rtcl9yZWZlcmVuY2Vfb25seSI6IHsKICAgICJjb25maWd1cmF0aW9uX3NoYTI1NiI6ICI2Nzg3NWY3ZTU0YzE0ZTNhYTExYjdkZWEzMTIwZWUwNTQxMTZiOTUxZWJkMjUwN2I4ZjUyNGMxYjllNDA3MTJhIiwKICAgICJ0ZW1wZXJhdHVyZSI6IDAuOCwKICAgICJwcmVzZXJ2ZV90b3AxIjogdHJ1ZSwKICAgICJ3YXJuaW5nIjogIlJlZmVyZW5jZSBvbmx5LiBOZXZlciBwb29sIHJlc3VsdHMgYWNyb3NzIHRoaXMgbGluZWFnZSBhbmQgaGlzdG9yaWNhbF92MDRfY29tbW9uX3Byb2plY3Rpb24uIgogIH0sCiAgImF1dGhvcml0eSI6IHsKICAgICJkZWZhdWx0X2FjdGlvbiI6ICJ0YWxvbl9tYXNzX3JpZ2h0IiwKICAgICJyZXNjdWVfbGFkZGVyIjogWwogICAgICAidGFsb25fbWFzc19sZWZ0IiwKICAgICAgInRhbG1fbWFzc19yaWdodF9zb2Z0IiwKICAgICAgImJhc2VsaW5lIgogICAgXSwKICAgICJmb3JtYWxfb3V0cHV0cyI6IFsKICAgICAgIktFRVBfREVGQVVMVCIsCiAgICAgICJTV0lUQ0hfUkVTQ1VFIiwKICAgICAgIkFCU1RBSU4iLAogICAgICAiQVNLIgogICAgXSwKICAgICJmYWlsX2Nsb3NlZCI6IHRydWUKICB9LAogICJ0ZW1wb3JhbF9vYnNlcnZhdGlvbiI6IHsKICAgICJjaGVja3BvaW50cyI6IFswLCAxNiwgMzIsIDY0XSwKICAgICJsb2NrX2FmdGVyX2ZyYWN0aW9uIjogMC43LAogICAgInByb2JlX3NhbXBsZXNfdG9rZW5zIjogZmFsc2UsCiAgICAicHJvYmVfbXV0YXRlc19nZW5lcmF0aW9uX3N0YXRlIjogZmFsc2UsCiAgICAicHJlZml4X3NvdXJjZSI6ICJ0YWxvbl9tYXNzX3JpZ2h0X3RyYWplY3RvcnkiCiAgfSwKICAiY2hlYXBfc2NyZWVuIjogewogICAgImVuYWJsZWQiOiB0cnVlLAogICAgImhlYWRyb29tX3Byb2JhYmlsaXR5X3RocmVzaG9sZCI6IDAuMiwKICAgICJtaW5pbXVtX29ic2VydmF0aW9ucyI6IDIKICB9LAogICJ2YWx1ZV9vZl9jb250aW51YXRpb24iOiB7CiAgICAibWluaW11bV9sY2JfYWZ0ZXJfY29zdCI6IDAuMCwKICAgICJoYXJtX3VjYjk1X2RlbHRhIjogMC4wMjUsCiAgICAiY29tcHV0ZV9jb3N0X3Blcl9wcm9iZSI6IDAuMDAyNSwKICAgICJyaXNrX3BlbmFsdHkiOiAxLjAKICB9LAogICJjYW1wYWlnbiI6IHsKICAgICJtb2RlbF9pZCI6ICJRd2VuL1F3ZW4zLjUtNEItQmFzZSIsCiAgICAiZGV2ZWxvcG1lbnRfcHJvbXB0cyI6IDQwLAogICAgInByb3RlY3RlZF9yZXNlcnZlX3Byb21wdHMiOiAyMCwKICAgICJzZWVkcyI6IFsxNCwgMjgsIDQyXSwKICAgICJhcm1zIjogWwogICAgICAiZml4ZWRfdG1yIiwKICAgICAgInNoYWRvd190bXJfbm9uX2ludGVyZmVyZW5jZSIsCiAgICAgICJzY3JlZW5fcGx1c19zaGFkb3ciLAogICAgICAicHJvYmVfYWxsX2Nvc3RfY29udHJvbCIsCiAgICAgICJjb3ZlcmFnZV9tYXRjaGVkX3JhbmRvbSIsCiAgICAgICJwb3N0aG9jX29yYWNsZV9kZXNjcmlwdGl2ZV9vbmx5IgogICAgXQogIH0sCiAgInByb21vdGlvbl9ydWxlIjogewogICAgInJlcXVpcmVzX3Byb3RlY3RlZF9yZXNlcnZlIjogdHJ1ZSwKICAgICJyZXF1aXJlc19ub25faW50ZXJmZXJlbmNlX2lkZW50aXR5IjogdHJ1ZSwKICAgICJyZXF1aXJlc19wb3NpdGl2ZV9wYWlyZWRfdXRpbGl0eV9hZnRlcl9jb3N0IjogdHJ1ZSwKICAgICJyZXF1aXJlc19oYXJtX3VjYjk1X3dpdGhpbl9kZWx0YSI6IHRydWUsCiAgICAiaGlzdG9yaWNhbF9hcmNoaXZlX2Nhbl9wcm9tb3RlIjogZmFsc2UKICB9Cn0K\",\"src/rdss_gate.py\":\"IiIiRmFpbC1jbG9zZWQgYXV0aG9yaXR5IGxvZ2ljIGZvciB0aGUgUkRTUyB0ZW1wb3JhbCBTSEFET1cgZ2F0ZS4KClRoaXMgbW9kdWxlIGlzIGRlbGliZXJhdGVseSBtb2RlbC1hZ25vc3RpYy4gIFRoZSBHUFUgYWRhcHRlciBzdXBwbGllcyB0ZW1wb3JhbApvYnNlcnZhdGlvbnMgYW5kIGZyb3plbiB2YWx1ZSBlc3RpbWF0ZXM7IHRoaXMgbW9kdWxlIGRlY2lkZXMgd2hldGhlciBhdXRob3JpdHkKbWF5IGxlYXZlIHRoZSBkZWZhdWx0IFRBTE9OIG1hc3MtcmlnaHQgYnJhbmNoLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGFzZGljdCwgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBJdGVyYWJsZSwgTWFwcGluZywgU2VxdWVuY2UKCgpLRUVQX0RFRkFVTFQgPSAiS0VFUF9ERUZBVUxUIgpTV0lUQ0hfUkVTQ1VFID0gIlNXSVRDSF9SRVNDVUUiCkFCU1RBSU4gPSAiQUJTVEFJTiIKQVNLID0gIkFTSyIKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBDYW5kaWRhdGVFc3RpbWF0ZToKICAgIGFjdGlvbjogc3RyCiAgICBleHBlY3RlZF9nYWluOiBmbG9hdAogICAgZ2Fpbl9sY2I5NTogZmxvYXQKICAgIGhhcm1fcHJvYmFiaWxpdHlfdWNiOTU6IGZsb2F0CiAgICBwcm9iZV9jb3N0OiBmbG9hdAogICAgb2JzZXJ2YXRpb25zOiBpbnQKCiAgICBkZWYgdmFsdWVfbGNiX2FmdGVyX2Nvc3Qoc2VsZiwgcmlza19wZW5hbHR5OiBmbG9hdCA9IDEuMCkgLT4gZmxvYXQ6CiAgICAgICAgcmV0dXJuICgKICAgICAgICAgICAgZmxvYXQoc2VsZi5nYWluX2xjYjk1KQogICAgICAgICAgICAtIGZsb2F0KHNlbGYucHJvYmVfY29zdCkKICAgICAgICAgICAgLSByaXNrX3BlbmFsdHkgKiBmbG9hdChzZWxmLmhhcm1fcHJvYmFiaWxpdHlfdWNiOTUpCiAgICAgICAgKQoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIEdhdGVDb25maWc6CiAgICBkZWZhdWx0X2FjdGlvbjogc3RyID0gInRhbG9uX21hc3NfcmlnaHQiCiAgICByZXNjdWVfbGFkZGVyOiB0dXBsZVtzdHIsIC4uLl0gPSAoCiAgICAgICAgInRhbG9uX21hc3NfbGVmdCIsCiAgICAgICAgInRhbG1fbWFzc19yaWdodF9zb2Z0IiwKICAgICAgICAiYmFzZWxpbmUiLAogICAgKQogICAgaGVhZHJvb21fcHJvYmFiaWxpdHlfdGhyZXNob2xkOiBmbG9hdCA9IDAuMjAKICAgIG1pbmltdW1fb2JzZXJ2YXRpb25zOiBpbnQgPSAyCiAgICBtaW5pbXVtX2xjYl9hZnRlcl9jb3N0OiBmbG9hdCA9IDAuMAogICAgaGFybV91Y2I5NV9kZWx0YTogZmxvYXQgPSAwLjAyNQogICAgcmlza19wZW5hbHR5OiBmbG9hdCA9IDEuMAogICAgZmFpbF9jbG9zZWQ6IGJvb2wgPSBUcnVlCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgR2F0ZURlY2lzaW9uOgogICAgb3V0Y29tZTogc3RyCiAgICBnZW5lcmF0aW9uX2FjdGlvbjogc3RyCiAgICBwcm9iZV9hY3Rpb246IHN0ciB8IE5vbmUKICAgIHJlYXNvbjogc3RyCiAgICBoZWFkcm9vbV9wcm9iYWJpbGl0eTogZmxvYXQKICAgIHNlbGVjdGVkX2VzdGltYXRlOiBNYXBwaW5nW3N0ciwgb2JqZWN0XSB8IE5vbmUgPSBOb25lCgogICAgZGVmIGFzX2RpY3Qoc2VsZikgLT4gZGljdFtzdHIsIG9iamVjdF06CiAgICAgICAgcmV0dXJuIGFzZGljdChzZWxmKQoKCmNsYXNzIFRlbXBvcmFsU2hhZG93R2F0ZToKICAgICIiIkF1dGhvcml0eSBnYXRlIHdpdGggYSBzdHJpY3QgZGVmYXVsdCBicmFuY2ggYW5kIGZyb3plbi1wb2xpY3kgYm91bmRhcnkuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNvbmZpZzogR2F0ZUNvbmZpZyB8IE5vbmUgPSBOb25lLCAqLCBwb2xpY3lfZnJvemVuOiBib29sID0gRmFsc2UpOgogICAgICAgIHNlbGYuY29uZmlnID0gY29uZmlnIG9yIEdhdGVDb25maWcoKQogICAgICAgIHNlbGYucG9saWN5X2Zyb3plbiA9IGJvb2wocG9saWN5X2Zyb3plbikKCiAgICBkZWYgc2NyZWVuKHNlbGYsIGhlYWRyb29tX3Byb2JhYmlsaXR5OiBmbG9hdCwgb2JzZXJ2YXRpb25zOiBpbnQpIC0+IGJvb2w6CiAgICAgICAgaWYgb2JzZXJ2YXRpb25zIDwgc2VsZi5jb25maWcubWluaW11bV9vYnNlcnZhdGlvbnM6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgIHJldHVybiBmbG9hdChoZWFkcm9vbV9wcm9iYWJpbGl0eSkgPj0gc2VsZi5jb25maWcuaGVhZHJvb21fcHJvYmFiaWxpdHlfdGhyZXNob2xkCgogICAgZGVmIGRlY2lkZSgKICAgICAgICBzZWxmLAogICAgICAgICosCiAgICAgICAgaGVhZHJvb21fcHJvYmFiaWxpdHk6IGZsb2F0LAogICAgICAgIGVzdGltYXRlczogU2VxdWVuY2VbQ2FuZGlkYXRlRXN0aW1hdGVdIHwgSXRlcmFibGVbQ2FuZGlkYXRlRXN0aW1hdGVdLAogICAgICAgIG9ic2VydmF0aW9uczogaW50LAogICAgICAgIGFsbG93X2FzazogYm9vbCA9IEZhbHNlLAogICAgKSAtPiBHYXRlRGVjaXNpb246CiAgICAgICAgY2ZnID0gc2VsZi5jb25maWcKCiAgICAgICAgaWYgbm90IHNlbGYuc2NyZWVuKGhlYWRyb29tX3Byb2JhYmlsaXR5LCBvYnNlcnZhdGlvbnMpOgogICAgICAgICAgICByZXR1cm4gR2F0ZURlY2lzaW9uKAogICAgICAgICAgICAgICAgS0VFUF9ERUZBVUxULAogICAgICAgICAgICAgICAgY2ZnLmRlZmF1bHRfYWN0aW9uLAogICAgICAgICAgICAgICAgTm9uZSwKICAgICAgICAgICAgICAgICJjaGVhcF9zY3JlZW5fbmVnYXRpdmVfb3JfaW1tYXR1cmUiLAogICAgICAgICAgICAgICAgZmxvYXQoaGVhZHJvb21fcHJvYmFiaWxpdHkpLAogICAgICAgICAgICApCgogICAgICAgIGVzdGltYXRlcyA9IHR1cGxlKGVzdGltYXRlcykKICAgICAgICBpZiBub3QgZXN0aW1hdGVzOgogICAgICAgICAgICByZXR1cm4gR2F0ZURlY2lzaW9uKAogICAgICAgICAgICAgICAgQVNLIGlmIGFsbG93X2FzayBlbHNlIEFCU1RBSU4sCiAgICAgICAgICAgICAgICBjZmcuZGVmYXVsdF9hY3Rpb24sCiAgICAgICAgICAgICAgICBOb25lLAogICAgICAgICAgICAgICAgInNjcmVlbl9wb3NpdGl2ZV9idXRfbm9fc2hhZG93X2V2aWRlbmNlIiwKICAgICAgICAgICAgICAgIGZsb2F0KGhlYWRyb29tX3Byb2JhYmlsaXR5KSwKICAgICAgICAgICAgKQoKICAgICAgICBvcmRlciA9IHtuYW1lOiBpbmRleCBmb3IgaW5kZXgsIG5hbWUgaW4gZW51bWVyYXRlKGNmZy5yZXNjdWVfbGFkZGVyKX0KICAgICAgICBhZG1pc3NpYmxlID0gWwogICAgICAgICAgICBpdGVtCiAgICAgICAgICAgIGZvciBpdGVtIGluIGVzdGltYXRlcwogICAgICAgICAgICBpZiBpdGVtLmFjdGlvbiBpbiBvcmRlcgogICAgICAgICAgICBhbmQgaXRlbS5vYnNlcnZhdGlvbnMgPj0gY2ZnLm1pbmltdW1fb2JzZXJ2YXRpb25zCiAgICAgICAgICAgIGFuZCBpdGVtLmhhcm1fcHJvYmFiaWxpdHlfdWNiOTUgPD0gY2ZnLmhhcm1fdWNiOTVfZGVsdGEKICAgICAgICAgICAgYW5kIGl0ZW0udmFsdWVfbGNiX2FmdGVyX2Nvc3QoY2ZnLnJpc2tfcGVuYWx0eSkKICAgICAgICAgICAgPiBjZmcubWluaW11bV9sY2JfYWZ0ZXJfY29zdAogICAgICAgIF0KICAgICAgICBhZG1pc3NpYmxlLnNvcnQoCiAgICAgICAgICAgIGtleT1sYW1iZGEgaXRlbTogKAogICAgICAgICAgICAgICAgLWl0ZW0udmFsdWVfbGNiX2FmdGVyX2Nvc3QoY2ZnLnJpc2tfcGVuYWx0eSksCiAgICAgICAgICAgICAgICBvcmRlcltpdGVtLmFjdGlvbl0sCiAgICAgICAgICAgICkKICAgICAgICApCgogICAgICAgIGlmIG5vdCBhZG1pc3NpYmxlOgogICAgICAgICAgICByZXR1cm4gR2F0ZURlY2lzaW9uKAogICAgICAgICAgICAgICAgS0VFUF9ERUZBVUxULAogICAgICAgICAgICAgICAgY2ZnLmRlZmF1bHRfYWN0aW9uLAogICAgICAgICAgICAgICAgTm9uZSwKICAgICAgICAgICAgICAgICJub19yZXNjdWVfY2xlYXJzX3ZhbHVlX2FuZF9oYXJtX2JvdW5kcyIsCiAgICAgICAgICAgICAgICBmbG9hdChoZWFkcm9vbV9wcm9iYWJpbGl0eSksCiAgICAgICAgICAgICkKCiAgICAgICAgYmVzdCA9IGFkbWlzc2libGVbMF0KICAgICAgICBpZiBub3Qgc2VsZi5wb2xpY3lfZnJvemVuOgogICAgICAgICAgICByZXR1cm4gR2F0ZURlY2lzaW9uKAogICAgICAgICAgICAgICAgS0VFUF9ERUZBVUxULAogICAgICAgICAgICAgICAgY2ZnLmRlZmF1bHRfYWN0aW9uLAogICAgICAgICAgICAgICAgYmVzdC5hY3Rpb24sCiAgICAgICAgICAgICAgICAic2hhZG93X29ubHlfcG9saWN5X25vdF9mcm96ZW4iLAogICAgICAgICAgICAgICAgZmxvYXQoaGVhZHJvb21fcHJvYmFiaWxpdHkpLAogICAgICAgICAgICAgICAgYXNkaWN0KGJlc3QpLAogICAgICAgICAgICApCgogICAgICAgIHJldHVybiBHYXRlRGVjaXNpb24oCiAgICAgICAgICAgIFNXSVRDSF9SRVNDVUUsCiAgICAgICAgICAgIGJlc3QuYWN0aW9uLAogICAgICAgICAgICBiZXN0LmFjdGlvbiwKICAgICAgICAgICAgImZyb3plbl9wb2xpY3lfcmVzY3VlX2NsZWFyc19hbGxfYm91bmRzIiwKICAgICAgICAgICAgZmxvYXQoaGVhZHJvb21fcHJvYmFiaWxpdHkpLAogICAgICAgICAgICBhc2RpY3QoYmVzdCksCiAgICAgICAgKQo=\",\"src/shadow_campaign.py\":\"...\",\"upstream/canonical_talm_talon.py\":\"...\",\"upstream/cfldr_plus_v0_4_repaired.py\":\"...\"}")
ROOT = Path('/kaggle/working/rdss_temporal_shadow_gate_v0_1')
for relative, encoded in EMBEDDED.items():
    target = ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(base64.b64decode(encoded))
print('Materialized', len(EMBEDDED), 'frozen files at', ROOT)


In [ ]:
import subprocess, sys
command = [
    sys.executable, str(ROOT / 'src' / 'shadow_campaign.py'),
    '--seed', str(SEED),
    '--limit', str(LIMIT),
    '--output-root', '/kaggle/working',
    '--development-prompt-ids', DEVELOPMENT_PROMPT_IDS,
]
if CAMPAIGN_PROMPT_IDS:
    command.extend(['--campaign-prompt-ids', CAMPAIGN_PROMPT_IDS])
if RUN_DEVELOPMENT_COUNTERFACTUALS:
    command.append('--counterfactuals')
subprocess.run(command, check=True)


In [ ]:
from IPython.display import display, JSON
run_dir = Path('/kaggle/working') / f'RDSS_TEMPORAL_SHADOW_V0_1_SEED_{SEED}'
manifest = json.loads((run_dir / 'manifest.json').read_text())
display(JSON(manifest))
assert manifest['non_interference_identity_pass'], manifest['identity_failures']
print('G1a non-interference gate passed.')
